In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

class StackingModel:
    def __init__(self):
        self.label_encoders = {}
        self.minmax_scaler = MinMaxScaler()
        self.std_scaler = StandardScaler()
        self.xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
        self.rf_model = RandomForestClassifier(random_state=42)
        self.lr_model = LogisticRegression(max_iter=1000, random_state=42)
        self.meta_learner = LogisticRegression(max_iter=1000, random_state=42)
        self.features_to_encode = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
        self.feature_order = None

    def preprocess(self, df, fit=False):
        df = df.copy()
        # Encode categorical features
        if fit:
            for col in self.features_to_encode:
                le = LabelEncoder()
                df[col] = le.fit_transform(df[col].astype(str))
                self.label_encoders[col] = le
        else:
            for col in self.features_to_encode:
                le = self.label_encoders[col]
                df[col] = le.transform(df[col].astype(str))

        # Chuẩn hóa các cột numeric cùng loại scaler
        cols_minmax = ['age', 'avg_glucose_level']
        cols_std = ['bmi']

        if fit:
            df[cols_minmax] = self.minmax_scaler.fit_transform(df[cols_minmax])
            df[cols_std] = self.std_scaler.fit_transform(df[cols_std])
        else:
            df[cols_minmax] = self.minmax_scaler.transform(df[cols_minmax])
            df[cols_std] = self.std_scaler.transform(df[cols_std])

        return df

    def fit(self, df, target_col='stroke'):
        df = self.preprocess(df, fit=True)
        x = df.drop(columns=[target_col])
        y = df[target_col]

        print("Phân phối nhãn trước xử lý:")
        print(y.value_counts())

        # Kiểm tra nếu mất cân bằng đáng kể thì mới thực hiện resampling
        ratio = y.value_counts().min() / y.value_counts().max()
        
        if ratio < 0.8:
            # Under-sampling
            try:
                under = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
                x, y = under.fit_resample(x, y)
                print("Đã under-sample")
            except ValueError as e:
                print("Under-sampling thất bại:", e)

            # Over-sampling
            try:
                over = SMOTE(sampling_strategy=1.0, random_state=42)
                x, y = over.fit_resample(x, y)
                print("Đã over-sample")
            except ValueError as e:
                print("Over-sampling thất bại:", e)
        else:
            print("Bộ dữ liệu đã cân bằng, bỏ qua under/over sampling.")

        print("Phân phối nhãn sau xử lý:")
        print(y.value_counts())

        self.feature_order = x.columns.tolist()
        x_train, _, y_train, _ = train_test_split(x, y, test_size=0.2, random_state=42)

        self.xgb_model.fit(x_train, y_train)
        self.rf_model.fit(x_train, y_train)
        self.lr_model.fit(x_train, y_train)

        xgb_prob = self.xgb_model.predict_proba(x_train)[:, 1]
        rf_prob = self.rf_model.predict_proba(x_train)[:, 1]
        lr_prob = self.lr_model.predict_proba(x_train)[:, 1]
        meta_X = np.column_stack((xgb_prob, rf_prob, lr_prob))

        self.meta_learner.fit(meta_X, y_train)
        print("Đã huấn luyện xong mô hình stacking.")

    def predict(self, df):
        df = self.preprocess(df, fit=False)
        df = df[self.feature_order]  # bảo đảm đúng thứ tự cột
        xgb_prob = self.xgb_model.predict_proba(df)[:, 1]
        rf_prob = self.rf_model.predict_proba(df)[:, 1]
        lr_prob = self.lr_model.predict_proba(df)[:, 1]
        meta_X = np.column_stack((xgb_prob, rf_prob, lr_prob))
        return self.meta_learner.predict(meta_X)

    def predict_proba(self, df):
        df = self.preprocess(df, fit=False)
        df = df[self.feature_order]
        xgb_prob = self.xgb_model.predict_proba(df)[:, 1]
        rf_prob = self.rf_model.predict_proba(df)[:, 1]
        lr_prob = self.lr_model.predict_proba(df)[:, 1]
        meta_X = np.column_stack((xgb_prob, rf_prob, lr_prob))
        return self.meta_learner.predict_proba(meta_X)[:, 1]

    def save(self, path='stacking_model.pkl'):
        joblib.dump(self, path)

    @staticmethod
    def load(path='stacking_model.pkl'):
        return joblib.load(path)

In [7]:
import pandas as pd
df = pd.read_csv(r'C:\GIT\stroke_prediction_repo\stroke_data.csv')
df.head(10)

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,1,16,0,0,0,3,0,111.93,32.2,2,0
1,1,33,0,0,1,1,1,82.83,25.4,0,0
2,1,5,0,0,0,0,0,160.83,17.8,0,0
3,1,3,0,0,0,0,1,59.05,16.6,0,0
4,0,63,1,0,1,4,1,228.20,37.7,2,0
5,0,2,0,0,0,0,0,89.72,17.8,0,0
6,0,62,0,1,1,1,1,124.37,28.3,2,0
7,1,4,0,0,0,0,0,62.48,19.9,0,0
8,0,22,0,0,0,3,1,130.34,22.0,2,0
9,1,51,0,0,1,3,1,63.61,42.3,0,0


In [13]:
print(df['stroke'].head(10))

0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
Name: stroke, dtype: int64


In [8]:
import joblib

model = joblib.load('stacking_model.pkl')

In [9]:
df_new = df.drop(columns='stroke')
df_new = df_new.head(10)

In [10]:
df_new

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status
0,1,16,0,0,0,3,0,111.93,32.2,2
1,1,33,0,0,1,1,1,82.83,25.4,0
2,1,5,0,0,0,0,0,160.83,17.8,0
3,1,3,0,0,0,0,1,59.05,16.6,0
4,0,63,1,0,1,4,1,228.20,37.7,2
5,0,2,0,0,0,0,0,89.72,17.8,0
6,0,62,0,1,1,1,1,124.37,28.3,2
7,1,4,0,0,0,0,0,62.48,19.9,0
8,0,22,0,0,0,3,1,130.34,22.0,2
9,1,51,0,0,1,3,1,63.61,42.3,0


In [11]:
pred = model.predict(df_new)
proba = model.predict_proba(df_new)

print("Dự đoán nhãn (0/1):")
print(pred)

print("\nXác suất dự đoán (label=1):")
print(proba)

Dự đoán nhãn (0/1):
[0 0 0 0 0 0 0 0 0 0]

Xác suất dự đoán (label=1):
[0.00110775 0.00134895 0.00108581 0.00111897 0.00201728 0.00108776
 0.00920057 0.00110651 0.00110033 0.00256602]
